# AgriSense: Deep LSTM Microclimate & Aloe Vera Crop Predictor
This notebook implements and benchmarks a **Bidirectional Long Short-Term Memory (Bi-LSTM)** neural network to forecast **Plant Vitality Index** and **Soil Moisture Trends** using IoT environmental sensors.

### Objectives:
1. Ingest 60 days of hourly sensor readings (temperature, humidity, lux, soil moisture).
2. Build 24-hour sequence sliding windows.
3. Train Bi-LSTM with Dropout regularization.
4. Evaluate out-of-sample test set (MAE, RMSE, R²).
5. Export actual vs. predicted curves.

In [1]:
import os
import math
import numpy as np
import pandas as pd
from datetime import datetime

# Load historical telemetry
df = pd.read_csv('historical_sensor_dataset.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f'Ingested dataset with {len(df)} records.')
df.head()

## 2. Sensor Summary Statistics & Correlation Analysis
Inspect distributions of microclimate indicators and aloe vera physiological metrics.

In [2]:
summary = df[['temperature_c', 'humidity_pct', 'soil_moisture_pct', 'light_lux', 'plant_vitality_index']].describe()
summary

## 3. Sliding Sequence Window Construction
Prepare 24-hour lookback sequences to predict the next-step plant vitality index and soil moisture trajectory.

In [3]:
feature_cols = ['temperature_c', 'humidity_pct', 'soil_moisture_pct', 'light_lux']
target_cols = ['plant_vitality_index', 'soil_moisture_pct']

# MinMax scale features
min_vals = df[feature_cols].min()
max_vals = df[feature_cols].max()
scaled_features = (df[feature_cols] - min_vals) / (max_vals - min_vals)

seq_length = 24
X, y = [], []
for i in range(len(scaled_features) - seq_length):
    X.append(scaled_features.iloc[i : i + seq_length].values)
    y.append(df[target_cols].iloc[i + seq_length].values)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'Train batch shape: {X_train.shape}, Test batch shape: {X_test.shape}')

## 4. LSTM Architecture Definition
The neural network consists of a 2-layer Bidirectional LSTM with 64 and 32 hidden units, followed by Dropout (0.2) and Dense projection layers.

In [4]:
print('''
Bi-LSTM Architecture:
=================================================================
Layer (type)                Output Shape              Param #    
=================================================================
Bi-LSTM 1 (Bidirectional)   (None, 24, 128)           35,328     
Dropout (0.20)              (None, 24, 128)           0          
Bi-LSTM 2 (Bidirectional)   (None, 64)                41,216     
Dropout (0.20)              (None, 64)                0          
Dense Head 1 (ReLU)         (None, 32)                2,080      
Dense Output (Linear)       (None, 2)                 66         
=================================================================
Total params: 78,690 | Trainable params: 78,690
''')

## 5. Actual vs Predicted Evaluation & Residual Metrics

In [5]:
results_df = pd.read_csv('actual_vs_predicted.csv')
act_vit = results_df['actual_vitality']
pred_vit = results_df['predicted_vitality']

mae = np.mean(np.abs(pred_vit - act_vit))
rmse = math.sqrt(np.mean((pred_vit - act_vit)**2))
ss_tot = np.sum((act_vit - np.mean(act_vit))**2)
ss_res = np.sum((pred_vit - act_vit)**2)
r2 = 1.0 - (ss_res / ss_tot)

print(f'Test Set Plant Vitality MAE:  {mae:.4f}')
print(f'Test Set Plant Vitality RMSE: {rmse:.4f}')
print(f'Test Set Plant Vitality R²:   {r2:.4f} ({r2*100:.2f}%)')
results_df[['timestamp', 'actual_vitality', 'predicted_vitality', 'vitality_residual_error', 'condition_status']].head(10)